In [ ]:
# Load packages
import os
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# ========================================================================================
# Calculate the interval between the biospecimen collection date and diagnosis date
# ========================================================================================

WORKSPACEBUCKET = os.environ["WORKSPACE_BUCKET"]

df1 = pd.read_csv(f"{WORKSPACEBUCKET}/data/covariates/biosample_collection_age.tsv", sep ='\t')

df2 = pd.read_csv(f"{WORKSPACEBUCKET}/data/covariates/diagnosis_age.tsv", sep ='\t')

df1 = df1[['person_id', 'biosample_collection_date']]

df2 = df2[~(df2['entry_datetime'].isna())][['person_id', 'entry_datetime']]

df = pd.merge(df1, df2, on='person_id')

df['entry_datetime'] = pd.to_datetime(df['entry_datetime'], errors='coerce').dt.tz_localize(None)

df['biosample_collection_date'] = pd.to_datetime(df['biosample_collection_date'], errors='coerce').dt.tz_localize(None)

df['interval'] = ((df['entry_datetime'] - df['biosample_collection_date']).dt.days / 365.25)

In [ ]:
# Load packages 
import matplotlib.pyplot as plt
import numpy as np

# ========================================================================================
# Report the interval between the biospecimen collection date and diagnosis date
# ========================================================================================

# Prepare data
intervals = df["interval"].dropna()

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)

# Histogram
ax.hist(
    intervals,
    bins=30,
    color="steelblue",
    edgecolor="white",
    linewidth=0.8,
    alpha=0.9
)

# Median reference line
median = intervals.median()
ax.axvline(
    median,
    color="darkred",
    linestyle="--",
    linewidth=1.8,
    label=f"Median = {median:.2f}"
)

# Labels and title
ax.set_title("Distribution of Intervals", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Interval", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)

# Styling
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=10)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(
    "interval_distribution.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
plt.close(fig)

print(df['interval'].describe())

print(f"Number of individuals who received a daignosis after WGS biosampling: {len(df[df['interval'] > 0])}")

In [ ]:
# Load packages 
import matplotlib.pyplot as plt
import numpy as np

# ========================================================================================
# Filter  based on the interval between the biospecimen collection date and diagnosis date
# ========================================================================================

WORKSPACEBUCKET = os.environ["WORKSPACE_BUCKET"]

eur = pd.read_csv(f"{WORKSPACEBUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/EUR-like_exome_qc.txt", sep ='\t')

df = df[df['person_id'].isin(eur['IID'])]

# Prepare data
intervals = df["interval"].dropna()

fig, ax = plt.subplots(figsize=(9, 5.5), dpi=150)

# Histogram
ax.hist(
    intervals,
    bins=30,
    color="steelblue",
    edgecolor="white",
    linewidth=0.8,
    alpha=0.9
)

# Median reference line
median = intervals.median()
ax.axvline(
    median,
    color="darkred",
    linestyle="--",
    linewidth=1.8,
    label=f"Median = {median:.2f}"
)

# Labels and title
ax.set_title("Distribution of Intervals", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Interval", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)

# Styling
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=10)
ax.legend(frameon=False, fontsize=10)

plt.tight_layout()
plt.savefig(
    "eur_interval_distribution.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
plt.close(fig)

print(df['interval'].describe())

df = df[df['interval'] > 0]

print(f"Number of individuals who received a daignosis after WGS biosampling: {len(df)}")

In [ ]:
# ========================================================================================
# Save the interval between the biospecimen collection date and diagnosis date
# ========================================================================================

WORKSPACEBUCKET = os.environ["WORKSPACE_BUCKET"]

df.to_csv(f"{WORKSPACEBUCKET}/data/covariates/daignosis_interval.tsv", 
                         sep ='\t',
                         header=False,
                         index=False)

df[['person_id']].to_csv(f"{WORKSPACEBUCKET}/data/covariates/daignosis_interval.samplelist", 
                         sep ='\t',
                         header=False,
                         index=False)

file = pd.read_csv(f"{WORKSPACEBUCKET}/data/saige_phenotype_files/aml_biosample_age_files/EUR_phenotype_file.txt", sep ='\t')

file = file[(file['aml_biosample_age'] == 0) | ((file['aml_biosample_age'] == 1) & (file['person_id'].isin(df['person_id'])))]

print(file['aml_biosample_age'].value_counts()) 

file[['person_id']].to_csv(f"{WORKSPACEBUCKET}/data/saige_phenotype_files/aml_biosample_age_files/samplelist.txt", 
                         sep ='\t',
                         header=False,
                         index=False)